<a href="https://colab.research.google.com/github/ascordero001-cell/enares-2024-crs04-ml/blob/main/notebooks/02%20bigquery/03_ENARES_2024_STAGE2_load_spss_metadata_to_bigquery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 03_ENARES_2024_STAGE2_load_spss_metadata_to_bigquery.ipynb
# Stage 2 - Load SPSS metadata to BigQuery
# ============================================================

!pip install -q google-cloud-bigquery pandas pyreadstat pandas-gbq pyarrow

from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import os
import hashlib
import pyreadstat

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()
LOCATION = "US"

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
SAV_DIR = f"{ROOT_DRIVE}/01BasesDatosPrimarias"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"

os.makedirs(LOG_DIR, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print("Using project:", PROJECT_ID)
print("Using LOG_DIR:", LOG_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.4 MB/s eta 0:00:00
Mounted at /content/drive
Enter your Google Cloud PROJECT_ID: enares-2024-crs04
Using project: enares-2024-crs04
Using LOG_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs


In [ ]:
# ============================================================
# 1. Load source file mapping from Notebook 2
# ============================================================

source_check_path = f"{LOG_DIR}/ENARES_2024_STAGE2_source_file_check.csv"

if not os.path.exists(source_check_path):
    raise FileNotFoundError(
        "Missing source file check CSV. Run Notebook 2 first."
    )

table_mapping = pd.read_csv(source_check_path)

required_cols = ["module", "chapter", "source_file", "target_table", "source_path"]

missing_cols = [c for c in required_cols if c not in table_mapping.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in source check CSV: {missing_cols}")

table_mapping["file_exists"] = table_mapping["source_path"].apply(os.path.exists)

if not table_mapping["file_exists"].all():
    print(table_mapping.to_string(index=False))
    raise FileNotFoundError("At least one .sav path from Notebook 2 no longer exists.")

print(table_mapping.to_string(index=False))

module chapter         source_file     target_table                                                                                                              source_path          raw_dataset  file_exists  file_size_bytes
 CRS04  CAP100 19_CRS04_CAP100.sav raw_crs04_cap100 /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1959/extracted/976-Modulo1959/19_CRS04_CAP100.sav enares2024_crs04_raw         True         15327888
 CRS04  CAP200 20_CRS04_CAP200.sav raw_crs04_cap200 /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1960/extracted/976-Modulo1960/20_CRS04_CAP200.sav enares2024_crs04_raw         True         14731025
 CRS04  CAP248 21_CRS04_CAP248.sav raw_crs04_cap248 /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1961/extracted/976-Modulo1961/21_CRS04_CAP248.sav enares2024_crs04_raw         True         16100226
 CRS04  CAP300 22_CRS04_CAP300.sav raw_crs04_cap300 /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDa

In [ ]:
# ============================================================
# 2. Extract SPSS metadata
# Critical point: use meta.variable_value_labels when available
# so labels stay associated with each variable.
# ============================================================

variables_rows = []
value_label_rows = []
missing_rows = []

for _, row in table_mapping.iterrows():
    print(f"Reading metadata: {row['source_file']}")

    _, meta = pyreadstat.read_sav(
        row["source_path"],
        metadataonly=True,
        apply_value_formats=False
    )

    chapter = row["chapter"]
    source_file = row["source_file"]
    target_table = row["target_table"]

    for var_name, var_label in zip(meta.column_names, meta.column_labels):
        variables_rows.append({
            "chapter": chapter,
            "source_file": source_file,
            "target_table": target_table,
            "variable_name": var_name,
            "variable_label": var_label,
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

    variable_value_labels = getattr(meta, "variable_value_labels", {}) or {}

    for var_name, labels in variable_value_labels.items():
        for value, label in labels.items():
            value_label_rows.append({
                "chapter": chapter,
                "source_file": source_file,
                "target_table": target_table,
                "variable_name": var_name,
                "value": str(value),
                "label": str(label),
                "checked_at_utc": datetime.now(timezone.utc).isoformat()
            })

    missing_user_values = getattr(meta, "missing_user_values", {}) or {}
    missing_ranges = getattr(meta, "missing_ranges", {}) or {}

    for var_name, values in missing_user_values.items():
        missing_rows.append({
            "chapter": chapter,
            "source_file": source_file,
            "target_table": target_table,
            "variable_name": var_name,
            "missing_type": "user_value",
            "missing_value": str(values),
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

    for var_name, ranges in missing_ranges.items():
        missing_rows.append({
            "chapter": chapter,
            "source_file": source_file,
            "target_table": target_table,
            "variable_name": var_name,
            "missing_type": "range",
            "missing_value": str(ranges),
            "checked_at_utc": datetime.now(timezone.utc).isoformat()
        })

variables_df = pd.DataFrame(variables_rows)
value_labels_df = pd.DataFrame(value_label_rows)

missing_df = pd.DataFrame(
    missing_rows,
    columns=[
        "chapter",
        "source_file",
        "target_table",
        "variable_name",
        "missing_type",
        "missing_value",
        "checked_at_utc"
    ]
)

print("variables:", len(variables_df))
print("value labels:", len(value_labels_df))
print("missing codes:", len(missing_df))

Reading metadata: 19_CRS04_CAP100.sav
Reading metadata: 20_CRS04_CAP200.sav
Reading metadata: 21_CRS04_CAP248.sav
Reading metadata: 22_CRS04_CAP300.sav
variables: 1299
value labels: 3317
missing codes: 0


In [ ]:
# ============================================================
# 3. Register source files and PDF references
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

source_rows = []

for _, row in table_mapping.iterrows():
    sav_path = Path(row["source_path"])

    # Look for PDFs near the .sav file and its parent folders.
    module_root = sav_path.parents[2] if len(sav_path.parents) >= 3 else sav_path.parent
    pdfs = list(module_root.rglob("*.pdf"))

    pdf_names = [p.name for p in pdfs]
    pdf_paths = [str(p) for p in pdfs]
    pdf_hashes = [sha256_file(str(p)) for p in pdfs]

    source_rows.append({
        "module": row["module"],
        "chapter": row["chapter"],
        "sav_file": row["source_file"],
        "sav_path": row["source_path"],
        "sav_sha256": sha256_file(row["source_path"]),
        "target_table": row["target_table"],
        "pdf_files_found": "; ".join(pdf_names),
        "pdf_paths_found": "; ".join(pdf_paths),
        "pdf_sha256_found": "; ".join(pdf_hashes),
        "questionnaire_pdf_file": None,
        "questionnaire_pdf_sha256": None,
        "questionnaire_pdf_drive_id": None,
        "variable_dictionary_pdf_file": None,
        "variable_dictionary_pdf_sha256": None,
        "variable_dictionary_pdf_drive_id": None,
        "pdf_dictionary_extracted_to_table": "no",
        "pdf_dictionary_extraction_notes": "PDFs registered from Drive filesystem when found; Drive IDs pending manual/API registration.",
        "checked_at_utc": datetime.now(timezone.utc).isoformat()
    })

source_files_df = pd.DataFrame(source_rows)

print(source_files_df.to_string(index=False))

module chapter            sav_file                                                                                                                 sav_path                                                       sav_sha256     target_table                                                                                                pdf_files_found                                                                                                                                                                                                                                                                                                          pdf_paths_found                                                                                                                   pdf_sha256_found questionnaire_pdf_file questionnaire_pdf_sha256 questionnaire_pdf_drive_id variable_dictionary_pdf_file variable_dictionary_pdf_sha256 variable_dictionary_pdf_drive_id pdf_dictionary_extracted_to_table      

In [ ]:
# ============================================================
# 4. Load metadata tables to BigQuery
# ============================================================

dataset_id = "enares2024_crs04_raw"

variables_df.to_gbq(
    f"{dataset_id}.metadata_crs04_variables",
    project_id=PROJECT_ID,
    if_exists="replace"
)

value_labels_df.to_gbq(
    f"{dataset_id}.metadata_crs04_value_labels",
    project_id=PROJECT_ID,
    if_exists="replace"
)

source_files_df.to_gbq(
    f"{dataset_id}.metadata_crs04_source_files",
    project_id=PROJECT_ID,
    if_exists="replace"
)

print("Loaded variables, value labels, and source files metadata.")

/tmp/ipykernel_7700/3510217047.py:7: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  variables_df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 9425.40it/s]
/tmp/ipykernel_7700/3510217047.py:13: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  value_labels_df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 8630.26it/s]
/tmp/ipykernel_7700/3510217047.py:19: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  source_files_df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 10010.27it/s]

Loaded variables, value labels, and source files metadata.


In [ ]:
# ============================================================
# 5. Load missing codes table
# If missing_df is empty, create a physical empty table with schema.
# ============================================================

missing_table_id = f"{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_missing_codes"

missing_schema = [
    bigquery.SchemaField("chapter", "STRING"),
    bigquery.SchemaField("source_file", "STRING"),
    bigquery.SchemaField("target_table", "STRING"),
    bigquery.SchemaField("variable_name", "STRING"),
    bigquery.SchemaField("missing_type", "STRING"),
    bigquery.SchemaField("missing_value", "STRING"),
    bigquery.SchemaField("checked_at_utc", "STRING"),
]

if len(missing_df) > 0:
    missing_df.to_gbq(
        "enares2024_crs04_raw.metadata_crs04_missing_codes",
        project_id=PROJECT_ID,
        if_exists="replace"
    )
    print("Loaded missing codes metadata.")
else:
    table = bigquery.Table(missing_table_id, schema=missing_schema)
    client.delete_table(missing_table_id, not_found_ok=True)
    client.create_table(table)
    print("No missing codes detected. Created empty physical metadata_crs04_missing_codes table with explicit schema.")

No missing codes detected. Created empty physical metadata_crs04_missing_codes table with explicit schema.


In [ ]:
# ============================================================
# 6. Create metadata inventory
# Required output: ENARES_2024_STAGE2_metadata_inventory.csv
# ============================================================

metadata_tables = [
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
]

metadata_inventory = []

for table_name in metadata_tables:
    full_table_id = f"{PROJECT_ID}.enares2024_crs04_raw.{table_name}"
    table = client.get_table(full_table_id)

    metadata_inventory.append({
        "table_name": table_name,
        "row_count": table.num_rows,
        "column_count": len(table.schema),
        "checked_at_utc": datetime.now(timezone.utc).isoformat()
    })

metadata_inventory = pd.DataFrame(metadata_inventory)

metadata_inventory_output = f"{LOG_DIR}/ENARES_2024_STAGE2_metadata_inventory.csv"
metadata_inventory.to_csv(metadata_inventory_output, index=False)

print(f"Metadata inventory saved to: {metadata_inventory_output}")
display(metadata_inventory)

Metadata inventory saved to: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_metadata_inventory.csv


,table_name,row_count,column_count,checked_at_utc
0,metadata_crs04_variables,1299,6,2026-05-28T00:01:00.662159+00:00
1,metadata_crs04_value_labels,3317,7,2026-05-28T00:01:00.851516+00:00
2,metadata_crs04_missing_codes,0,7,2026-05-28T00:01:01.008343+00:00
3,metadata_crs04_source_files,4,18,2026-05-28T00:01:01.154433+00:00


In [ ]:
# ============================================================
# 7. Final acceptance check
# ============================================================

expected_tables = {
    "metadata_crs04_variables",
    "metadata_crs04_value_labels",
    "metadata_crs04_missing_codes",
    "metadata_crs04_source_files",
}

actual_tables = set(metadata_inventory["table_name"])

if not expected_tables.issubset(actual_tables):
    raise RuntimeError("Not all required metadata tables were created.")

if not os.path.exists(metadata_inventory_output):
    raise FileNotFoundError("Required metadata inventory CSV was not created.")

print("Notebook 3 completed successfully.")
print("Required output created:")
print("ENARES_2024_STAGE2_metadata_inventory.csv")

Notebook 3 completed successfully.
Required output created:
ENARES_2024_STAGE2_metadata_inventory.csv


# Supplementary checks

In [1]:
!pip install -q google-cloud-bigquery pandas

from google.colab import auth, drive
from google.cloud import bigquery
import pandas as pd
import os

auth.authenticate_user()
drive.mount("/content/drive")

PROJECT_ID = input("Enter your Google Cloud PROJECT_ID: ").strip()
LOCATION = "US"

ROOT_DRIVE = "/content/drive/MyDrive/ENARES_2024_PROJECT"
LOG_DIR = f"{ROOT_DRIVE}/05Resultados/logs"

os.makedirs(LOG_DIR, exist_ok=True)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

Mounted at /content/drive
Enter your Google Cloud PROJECT_ID: enares-2024-crs04


In [2]:
# 12. Control C3 variables inside CRS04

c3_sql = f"""
SELECT chapter, source_file, variable_name, variable_label
FROM `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_variables`
WHERE STARTS_WITH(variable_name, "C3")
ORDER BY chapter, variable_name
"""

c3_variables = client.query(c3_sql).result().to_dataframe()

c3_output = f"{LOG_DIR}/ENARES_2024_STAGE2_c3_variables_in_crs04_check.csv"
c3_variables.to_csv(c3_output, index=False)

print(f"Saved: {c3_output}")
display(c3_variables.head(50))

Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_c3_variables_in_crs04_check.csv


,chapter,source_file,variable_name,variable_label
0,CAP100,19_CRS04_CAP100.sav,C3ANIO,AÑO DE ESTUDIO
1,CAP100,19_CRS04_CAP100.sav,C3P102_ANIO,102. AÑO DE NACIMIENTO
2,CAP100,19_CRS04_CAP100.sav,C3P102_MES,102. MES DE NACIMIENTO
3,CAP100,19_CRS04_CAP100.sav,C3P103EDAD,103. ¿CUÁNTOS AÑOS TIENES?
4,CAP100,19_CRS04_CAP100.sav,C3P104,104. ¿VIVES EN ESTE DISTRITO DE……………………………?
5,CAP100,19_CRS04_CAP100.sav,C3P105,105. ¿EL LUGAR DONDE VIVES ES:
6,CAP100,19_CRS04_CAP100.sav,C3P106,106. ¿TIENES MAMÁ?
7,CAP100,19_CRS04_CAP100.sav,C3P110,110. ¿TIENES PAPÁ?
8,CAP100,19_CRS04_CAP100.sav,C3P114_PERS,"114. EN TU CASA, INCLUYÉNDOTE ¿CUÁNTAS PERSONA..."
9,CAP100,19_CRS04_CAP100.sav,C3P114_VS,114. VIVO SOLA|O


In [3]:
# 13. Prepare key-validation SQL for Stage 3 only
# Do not execute merge in Stage 2

key_preview_sql = f"""
-- CONSULTA PARA FASE 03. NO CREA TABLAS CLEANED EN STAGE 2.

SELECT "CAP100" AS tabla,
       COUNT(*) AS total_rows,
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID)) AS distinct_key
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100`

UNION ALL
SELECT "CAP200",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap200`

UNION ALL
SELECT "CAP248",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap248`

UNION ALL
SELECT "CAP300",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap300`
"""

key_output = f"{LOG_DIR}/ENARES_2024_STAGE2_key_validation_preview_for_stage3.sql"

with open(key_output, "w", encoding="utf-8") as f:
    f.write(key_preview_sql)

print(f"Saved: {key_output}")
print(key_preview_sql)

Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_key_validation_preview_for_stage3.sql

-- CONSULTA PARA FASE 03. NO CREA TABLAS CLEANED EN STAGE 2.

SELECT "CAP100" AS tabla,
       COUNT(*) AS total_rows,
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID)) AS distinct_key
FROM `enares-2024-crs04.enares2024_crs04_raw.raw_crs04_cap100`

UNION ALL
SELECT "CAP200",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `enares-2024-crs04.enares2024_crs04_raw.raw_crs04_cap200`

UNION ALL
SELECT "CAP248",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `enares-2024-crs04.enares2024_crs04_raw.raw_crs04_cap248`

UNION ALL
SELECT "CAP300",
       COUNT(*),
       COUNT(DISTINCT STRUCT(ID, COLEGIAL_ID))
FROM `enares-2024-crs04.enares2024_crs04_raw.raw_crs04_cap300`



In [4]:
# 14. Save FLOAT64 key warning SQL for Stage 3

float_key_check_sql = f"""
-- USAR EN FASE 03 SI ID O COLEGIAL_ID LLEGAN COMO FLOAT64.
-- Esta consulta NO convierte llaves. Solo prepara la validacion.

SELECT
  "raw_crs04_cap100" AS tabla,
  COUNTIF(ID IS NULL) AS id_nulls,
  COUNTIF(COLEGIAL_ID IS NULL) AS colegial_id_nulls,
  COUNTIF(ID IS NOT NULL AND ID != FLOOR(ID)) AS id_with_decimals,
  COUNTIF(COLEGIAL_ID IS NOT NULL AND COLEGIAL_ID != FLOOR(COLEGIAL_ID)) AS colegial_id_with_decimals
FROM `{PROJECT_ID}.enares2024_crs04_raw.raw_crs04_cap100`
"""

float_output = f"{LOG_DIR}/ENARES_2024_STAGE2_float64_key_warning_for_stage3.sql"

with open(float_output, "w", encoding="utf-8") as f:
    f.write(float_key_check_sql)

print(f"Saved: {float_output}")
print(float_key_check_sql)

Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_float64_key_warning_for_stage3.sql

-- USAR EN FASE 03 SI ID O COLEGIAL_ID LLEGAN COMO FLOAT64.
-- Esta consulta NO convierte llaves. Solo prepara la validacion.

SELECT
  "raw_crs04_cap100" AS tabla,
  COUNTIF(ID IS NULL) AS id_nulls,
  COUNTIF(COLEGIAL_ID IS NULL) AS colegial_id_nulls,
  COUNTIF(ID IS NOT NULL AND ID != FLOOR(ID)) AS id_with_decimals,
  COUNTIF(COLEGIAL_ID IS NOT NULL AND COLEGIAL_ID != FLOOR(COLEGIAL_ID)) AS colegial_id_with_decimals
FROM `enares-2024-crs04.enares2024_crs04_raw.raw_crs04_cap100`



In [5]:
# 15. Document preservation of complex survey design variables

design_vars = ["CCDD", "ID", "ID_AULA"]  # Add official weight variable once confirmed.

design_sql = f"""
SELECT chapter, source_file, variable_name, variable_label
FROM `{PROJECT_ID}.enares2024_crs04_raw.metadata_crs04_variables`
WHERE variable_name IN UNNEST(@design_vars)
ORDER BY variable_name, chapter
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter("design_vars", "STRING", design_vars)
    ]
)

design_var_check = client.query(design_sql, job_config=job_config).result().to_dataframe()

design_output = f"{LOG_DIR}/ENARES_2024_STAGE2_design_variables_preservation_check.csv"
design_var_check.to_csv(design_output, index=False)

print(f"Saved: {design_output}")
display(design_var_check)

Saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_STAGE2_design_variables_preservation_check.csv


,chapter,source_file,variable_name,variable_label
0,CAP100,19_CRS04_CAP100.sav,CCDD,CÓDIGO DE DEPARTAMENTO
1,CAP200,20_CRS04_CAP200.sav,CCDD,CÓDIGO DE DEPARTAMENTO
2,CAP248,21_CRS04_CAP248.sav,CCDD,CÓDIGO DE DEPARTAMENTO
3,CAP300,22_CRS04_CAP300.sav,CCDD,CÓDIGO DE DEPARTAMENTO
4,CAP100,19_CRS04_CAP100.sav,ID,IDENTIFICACIÓN INFORMÁTICA
5,CAP200,20_CRS04_CAP200.sav,ID,IDENTIFICACIÓN INFORMÁTICA
6,CAP248,21_CRS04_CAP248.sav,ID,IDENTIFICACIÓN INFORMÁTICA
7,CAP300,22_CRS04_CAP300.sav,ID,IDENTIFICACIÓN INFORMÁTICA
